## Using TOSLA for the Analysis of Customer Agreements

#### Three Levels of Analysis with TOSLA

To evaluate the utility of representing Service Level Agreements (SLAs) as machine-readable contracts, we propose three complementary levels of analysis. This approach not only supports the full automation of the agreement lifecycle but also enables customers and providers to automatically analyse contractual clauses:

1. **Competency Question Validation** – Verify whether the model can successfully respond to the defined 8 Competency Questions, designed to extract key information and validate the expressiveness of the SLA representation.  
2. **Fairness Assessment** – Detect the presence of potentially unfair contractual terms by applying the queries proposed by the authors of [TOSL](https://w3id.org/tosl/). This analysis specifically targets 8 types of potentially unfair clauses, each representing a different category of risk or imbalance in the agreement.  
3. **Normative Analysis** – Identify the obligations, permissions, and prohibitions of the parties involved in the SLA.  

The normative analysis and the detection of potentially unfair clauses can be performed on the entire Customer Agreement (CA), rather than on the SLA in isolation. In this context, the agreement can be treated as a single contractual entity composed of multiple documents. For example, the ToS together with the SLA.

As a case study, we will demonstrate this approach by applying **TOSLA** to the **Alibaba Cloud Customer Agreement**, combining its **ToS** and **SLA** to illustrate how normative elements can be extracted and potentially unfair clauses automatically detected.

This capability represents the **fundamental value of TOSLA**: by enabling the modelling of SLAs using the ODRL language, it is possible to represent various types of policies in a formal and interoperable manner. As a result, SPARQL queries can be applied directly to these representations, enabling automated reasoning and analysis across different contractual dimensions.


---
#### System Requirements

In [1]:
! pip install -r ../requirements.txt

#### Auxiliary Function to Execute SPARQL Queries

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from tabulate import tabulate
from rdflib import Graph

def tabulate_result(results):
    rows = [row.asdict() for row in results]
    df = pd.DataFrame(rows)
    print(tabulate(df, headers=df.columns))

def run_sparql_query(file_path, query_file, file_format="ttl"):
    g = Graph()
    g.parse(file_path, format=file_format)

    with open(query_file, "r") as f:
        query = f.read()

    results = g.query(query)
    tabulate_result(results)

file_format = "ttl" 

---

## TOSLA Validator

The validator, located at `validator/agreement_shape.ttl`, employs the [Shapes Constraint Language (SHACL)](https://www.w3.org/TR/shacl/) to verify semantic constraints and ensure compliance with the **ODRL**, **TOSL**, and **TOSLA** specifications.

Before executing the Competency Questions (CQs), performing the obligation and rights analysis, or conducting the detection of potentially unfair clauses, the validator is run to assess whether the TTL files conform to the required specifications.

Next, the validator will be executed on each of the modeled agreements located in the `example/` directory. The final execution is performed on a dedicated testing file to demonstrate the output produced by the validator when errors are detected.


- Amazon EC2 SLA

In [3]:
from tosla_validator import validate_tosla_ttl

file_path = "../examples/amazon/amazon_ec2_sla.ttl"
with open(file_path, "r", encoding="utf-8") as f:
    ttl_content = f.read()

result = validate_tosla_ttl(ttl_content, "../validator/agreement_shape.ttl")
result

{'syntax_valid': True,
 'conforms': True,
 'validation_time_sec': 0.31946277618408203,
 'violations': []}

- Cedalo SLA

In [5]:
from tosla_validator import validate_tosla_ttl

file_path = "../examples/cedalo/cedalo_sla_2025.ttl"
with open(file_path, "r", encoding="utf-8") as f:
    ttl_content = f.read()

result = validate_tosla_ttl(ttl_content, "../validator/agreement_shape.ttl")
result

{'syntax_valid': True,
 'conforms': True,
 'validation_time_sec': 1.0774130821228027,
 'violations': []}

- Alibaba SLA

In [6]:
from tosla_validator import validate_tosla_ttl

file_path = "../examples/alibaba/alibaba_sla.ttl"
with open(file_path, "r", encoding="utf-8") as f:
    ttl_content = f.read()
    
result = validate_tosla_ttl(ttl_content, "../validator/agreement_shape.ttl")
result

{'syntax_valid': True,
 'conforms': True,
 'validation_time_sec': 0.3076438903808594,
 'violations': []}

- Alibaba ToS

In [23]:
from tosla_validator import validate_tosla_ttl

file_path = "../examples/alibaba/alibaba_tos.ttl"
with open(file_path, "r", encoding="utf-8") as f:
    ttl_content = f.read()
    
result = validate_tosla_ttl(ttl_content, "../validator/agreement_shape.ttl")
result

{'syntax_valid': True,
 'conforms': True,
 'validation_time_sec': 0.18213295936584473,
 'violations': []}

- SLA modifies for testing with errors

In [8]:
from tosla_validator import validate_tosla_ttl

file_path = "../examples/SLA_Modified_For_Testing.ttl"
with open(file_path, "r", encoding="utf-8") as f:
    ttl_content = f.read()
    
result = validate_tosla_ttl(ttl_content, "../validator/agreement_shape.ttl")
result

{'syntax_valid': True,
 'conforms': False,
 'validation_time_sec': 0.08215594291687012,
 'violations': [{'severity': 'http://www.w3.org/ns/shacl#Violation',
   'sourceShape': 'n7d80787fa6b24c25b6ad0905a620e88cb3',
   'sourceConstraintComponent': 'http://www.w3.org/ns/shacl#MinCountConstraintComponent',
   'focusNode': 'http://example.com/compensation15_SPlanSingleNode',
   'valueNode': '',
   'resultPath': 'http://www.w3.org/ns/odrl/2/assignee',
   'message': 'The assignee must be a tosl:Provider, tosl:Customer, or tosl:BusinessCustomer.'},
  {'severity': 'http://www.w3.org/ns/shacl#Violation',
   'sourceShape': 'n7d80787fa6b24c25b6ad0905a620e88cb6',
   'sourceConstraintComponent': 'http://www.w3.org/ns/shacl#OrConstraintComponent',
   'focusNode': 'http://example.com/commitmentSPlanSingleNode',
   'valueNode': 'n7d80787fa6b24c25b6ad0905a620e88cb7',
   'resultPath': 'http://www.w3.org/ns/odrl/2/action',
   'message': 'The value of the action node must be a valid action from the approve

---
## Competency Questions

In this section, we execute the defined **8 Competency Questions (CQs)** to extract relevant information from the SLAs. Each question will be applied to the SLAs of Amazon, Cedalo and Alibaba.


### CQ1: Which **services** are governed by the SLA?

- Amazon EC2 SLA

In [24]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ1_services.rq" 
run_sparql_query(file_path, query_file, file_format)

    agreement                        service                                serviceDescription
--  -------------------------------  -------------------------------------  -----------------------------------------------------------------
 0  http://example.com/amazonEC2SLA  http://example.com/ec2RegionService    EC2 instances deployed across multiple AZs within a single region
 1  http://example.com/amazonEC2SLA  http://example.com/ec2InstanceService  Single Amazon EC2 instance


- Cedalo SLA

In [26]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/competency_questions/CQ1_services.rq" 
run_sparql_query(file_path, query_file, file_format)

    agreement                     service                                          serviceDescription
--  ----------------------------  -----------------------------------------------  ------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 0  http://example.com/cedaloSLA  http://example.com/serviceSPlanSingleNode        Single Node service for Plan S: provides a standalone Pro Mosquitto MQTT broker, ideal for basic IoT deployments.
 1  http://example.com/cedaloSLA  http://example.com/serviceSPlanHighAvailability  High Availability for Plan S: connects and synchronizes multiple instances of Pro Mosquitto MQTT broker, ensuring operational continuity in case of node failure.
 2  http://example.com/cedaloSLA  http://example.com/serviceMPlanSingleNode        Single Node service for Plan M: dedicated Pro Mosquitto MQTT broker for medium-complexity IoT environments.
 3  http://ex

- Alibaba SLA

In [28]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ1_services.rq" 
run_sparql_query(file_path, query_file, file_format)

    agreement                         service                                 serviceDescription
--  --------------------------------  --------------------------------------  ------------------------------------------------------------------------------------------------------------
 0  http://example.com/alibabaECSSLA  http://example.com/instanceEcsService   Alibaba Cloud Elastic Compute Service (ECS) instance offering covered by this SLA.
 1  http://example.com/alibabaECSSLA  http://example.com/multiZoneEcsService  Alibaba Cloud ECS instances deployed in multiple zones within the same region, subject to higher uptime SLA.


### CQ2: Which **quality of service** levels does a service deliver?

- Amazon EC2 SLA

In [30]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ2_slo.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                commitment                                    description                                                                                                SLI                        operator                          objectiveValue
--  -------------------------------------  --------------------------------------------  ---------------------------------------------------------------------------------------------------------  -------------------------  --------------------------------  ----------------
 0  http://example.com/ec2InstanceService  http://example.com/commitmentEC2Instance      AWS must provide at least 99.5% monthly uptime for single EC2 instance                                     Monthly Uptime Percentage  http://www.w3.org/ns/odrl/2/gteq  99.5
 1  http://example.com/ec2InstanceService  http://example.com/commitmentPerInstanceHour  AWS must ensure that each EC2 instance is unavailable no more than 6 minutes within any given clock hou

- Cedalo SLA

In [31]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/competency_questions/CQ2_slo.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                          commitment                                          description                                                 SLI                        operator                            objectiveValue
--  -----------------------------------------------  --------------------------------------------------  ----------------------------------------------------------  -------------------------  --------------------------------  ----------------
 0  http://example.com/serviceEnterprisePlan         http://example.com/commitmentEnterprisePlan         Cedalo must provide an uptime percentage of at least 99.9%  Monthly Uptime Percentage  http://www.w3.org/ns/odrl/2/gteq              99.9
 1  http://example.com/serviceLPlanHighAvailability  http://example.com/commitmentLPlanHighAvailability  Cedalo must provide an uptime percentage of at least 99.9%  Monthly Uptime Percentage  http://www.w3.org/ns/odrl/2/gteq              99.9
 2  http://example.com/servi

- Alibaba SLA

In [32]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ2_slo.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                 commitment                                    description                                                               SLI                                                         operator                            objectiveValue
--  --------------------------------------  --------------------------------------------  ------------------------------------------------------------------------  ----------------------------------------------------------  --------------------------------  ----------------
 0  http://example.com/instanceEcsService   http://example.com/uptimeCommitmentInstance   Alibaba must provide at least 99.975% uptime per instance.                Monthly Uptime Percentage - Instance Unavailable            http://www.w3.org/ns/odrl/2/gteq            99.975
 1  http://example.com/multiZoneEcsService  http://example.com/uptimeCommitmentMultiZone  Alibaba must provide at least 99.995% uptime for multi-zone deployments.  Monthly Upt

### CQ3: Which particular **properties of a service** are guaranteed to have certain values?

- Amazon EC2 SLA

In [33]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ3_sli.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                SLI
--  -------------------------------------  ------------------------------------------------------
 0  http://example.com/ec2InstanceService  http://example.com/monthlyUptimeMetric
 1  http://example.com/ec2InstanceService  http://example.com/unavailabilityDurationPerHourMetric
 2  http://example.com/ec2RegionService    http://example.com/monthlyUptimeMetric


- Cedalo SLA

In [34]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/competency_questions/CQ3_sli.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                          SLI
--  -----------------------------------------------  --------------------------------------
 0  http://example.com/serviceEnterprisePlan         http://example.com/monthlyUptimeMetric
 1  http://example.com/serviceLPlanHighAvailability  http://example.com/monthlyUptimeMetric
 2  http://example.com/serviceLPlanSingleNode        http://example.com/monthlyUptimeMetric
 3  http://example.com/serviceMPlanHighAvailability  http://example.com/monthlyUptimeMetric
 4  http://example.com/serviceMPlanSingleNode        http://example.com/monthlyUptimeMetric
 5  http://example.com/serviceSPlanHighAvailability  http://example.com/monthlyUptimeMetric
 6  http://example.com/serviceSPlanSingleNode        http://example.com/monthlyUptimeMetric


- Alibaba SLA

In [35]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ3_sli.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                 SLI
--  --------------------------------------  --------------------------------------
 0  http://example.com/instanceEcsService   http://example.com/monthlyUptimeMetric
 1  http://example.com/multiZoneEcsService  http://example.com/monthlyUptimeMetric


### CQ4: Which **compensations** are offered if the guaranteed value of a property is not honored?

- Amazon EC2 SLA

In [36]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ4_compensations.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                compensation                                   SLI                                               interval                          credit
--  -------------------------------------  ---------------------------------------------  ------------------------------------------------  ------------------------------  --------
 0  http://example.com/ec2InstanceService  http://example.com/compensationEC2Instance10   http://example.com/monthlyUptimePercentage        value ≥ 99.0 AND value < 99.5         10
 1  http://example.com/ec2InstanceService  http://example.com/compensationEC2Instance30   http://example.com/monthlyUptimePercentage        value ≥ 95.0 AND value < 99.0         30
 2  http://example.com/ec2InstanceService  http://example.com/compensationEC2Instance100  http://example.com/monthlyUptimePercentage        value < 99.5                         100
 3  http://example.com/ec2InstanceService  http://example.com/compensatePerHour           http:

- Cedalo SLA

In [37]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/competency_questions/CQ4_compensations.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                          compensation                                              SLI                                         interval                           credit
--  -----------------------------------------------  --------------------------------------------------------  ------------------------------------------  -------------------------------  --------
 0  http://example.com/serviceEnterprisePlan         http://example.com/compensation7_5_EnterprisePlan         http://example.com/monthlyUptimePercentage  value ≥ 98.99 AND value < 99.99       7.5
 1  http://example.com/serviceEnterprisePlan         http://example.com/compensation15_EnterprisePlan          http://example.com/monthlyUptimePercentage  value ≥ 89.99 AND value < 98.99      15
 2  http://example.com/serviceEnterprisePlan         http://example.com/compensation25_EnterprisePlan          http://example.com/monthlyUptimePercentage  value < 89.99                        25
 3  http://example.

- Alibaba SLA

In [38]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ4_compensations.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                 compensation                                 SLI                                                             interval                           credit
--  --------------------------------------  -------------------------------------------  --------------------------------------------------------------  -------------------------------  --------
 0  http://example.com/instanceEcsService   http://example.com/compensationInstance10    http://example.com/monthlyUptimePercentageInstanceUnavailable   value ≥ 99.0 AND value < 99.975        10
 1  http://example.com/instanceEcsService   http://example.com/compensationInstance25    http://example.com/monthlyUptimePercentageInstanceUnavailable   value ≥ 95.0 AND value < 99.0          25
 2  http://example.com/instanceEcsService   http://example.com/compensationInstance100   http://example.com/monthlyUptimePercentageInstanceUnavailable   value < 95.0                          100
 3  http://example.com/mu

### CQ5: Who is the **responsible party** for enforcing the guaranteed service level values?

- Amazon EC2 SLA

In [39]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ5_responsable.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                commitment                                    assignee                   assigneeName
--  -------------------------------------  --------------------------------------------  -------------------------  -------------------
 0  http://example.com/ec2InstanceService  http://example.com/commitmentEC2Instance      http://example.com/amazon  Amazon Web Services
 1  http://example.com/ec2InstanceService  http://example.com/commitmentPerInstanceHour  http://example.com/amazon  Amazon Web Services
 2  http://example.com/ec2RegionService    http://example.com/commitmentEC2Region        http://example.com/amazon  Amazon Web Services


- Cedalo SLA

In [40]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/competency_questions/CQ5_responsable.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                          commitment                                          assignee                   assigneeName
--  -----------------------------------------------  --------------------------------------------------  -------------------------  --------------------
 0  http://example.com/serviceEnterprisePlan         http://example.com/commitmentEnterprisePlan         http://example.com/cedalo  Cedalo MQTT Platform
 1  http://example.com/serviceLPlanHighAvailability  http://example.com/commitmentLPlanHighAvailability  http://example.com/cedalo  Cedalo MQTT Platform
 2  http://example.com/serviceLPlanSingleNode        http://example.com/commitmentLPlanSingleNode        http://example.com/cedalo  Cedalo MQTT Platform
 3  http://example.com/serviceMPlanHighAvailability  http://example.com/commitmentMPlanHighAvailability  http://example.com/cedalo  Cedalo MQTT Platform
 4  http://example.com/serviceMPlanSingleNode        http://example.com/commitmentMPlanSin

- Alibaba SLA

In [41]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ5_responsable.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                 commitment                                    assignee                    assigneeName
--  --------------------------------------  --------------------------------------------  --------------------------  --------------
 0  http://example.com/instanceEcsService   http://example.com/uptimeCommitmentInstance   http://example.com/alibaba  Alibaba Cloud
 1  http://example.com/multiZoneEcsService  http://example.com/uptimeCommitmentMultiZone  http://example.com/alibaba  Alibaba Cloud


### CQ6: Who is the **responsible party for monitoring and computing** the guaranteed values?

- Amazon EC2 SLA

In [42]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ6_monitoring.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                          liabilityType                               liableParty                  liabilityDescription
--  --------------------------------------------  ------------------------------------------  ---------------------------  -------------------------------------------------------------------------------------------------------------------------------------------------------------------
 0  http://example.com/commitmentEC2Instance      https://w3id.org/tosla/conditionEvaluation  http://example.com/customer  The customer is responsible for monitoring and compiling all information regarding service unavailability in order to claim compensation.
 1  http://example.com/commitmentEC2Instance      https://w3id.org/tosla/metricComputation    http://example.com/customer  The customer is responsible for calculating the total downtime affecting their services, in order to determine whether the guaranteed service levels have been met.
 2  http://example.co

- Cedalo SLA

In [43]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/competency_questions/CQ6_monitoring.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                                liabilityType                               liableParty                                liabilityDescription
--  --------------------------------------------------  ------------------------------------------  -----------------------------------------  -------------------------------------------------------------------------------------------------------------------------------------------------------------------
 0  http://example.com/commitmentEnterprisePlan         https://w3id.org/tosla/conditionEvaluation  http://example.com/customerEnterprisePlan  The customer is responsible for monitoring and compiling all information regarding service unavailability in order to claim compensation.
 1  http://example.com/commitmentEnterprisePlan         https://w3id.org/tosla/metricComputation    http://example.com/customerEnterprisePlan  The customer is responsible for calculating the total downtime affecting their services, in order to dete

- Alibaba SLA

In [44]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ6_monitoring.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                          liabilityType                               liableParty                  liabilityDescription
--  --------------------------------------------  ------------------------------------------  ---------------------------  -------------------------------------------------------------------------------------------------------------------------------------------------------------------
 0  http://example.com/uptimeCommitmentInstance   https://w3id.org/tosla/conditionEvaluation  http://example.com/customer  The customer is responsible for monitoring and compiling all information regarding service unavailability in order to claim compensation.
 1  http://example.com/uptimeCommitmentInstance   https://w3id.org/tosla/metricComputation    http://example.com/customer  The customer is responsible for calculating the total downtime affecting their services, in order to determine whether the guaranteed service levels have been met.
 2  http://example.co

### CQ7: During which **period of time** a guarantee is offered?

- Amazon EC2 SLA

In [45]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ7_period_time.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                duty                                          duration
--  -------------------------------------  --------------------------------------------  ----------
 0  http://example.com/ec2InstanceService  http://example.com/commitmentEC2Instance      P30D
 1  http://example.com/ec2InstanceService  http://example.com/commitmentPerInstanceHour  PT1H
 2  http://example.com/ec2RegionService    http://example.com/commitmentEC2Region        P30D


- Cedalo SLA

In [46]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/competency_questions/CQ7_period_time.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                          duty                                                duration
--  -----------------------------------------------  --------------------------------------------------  ----------
 0  http://example.com/serviceEnterprisePlan         http://example.com/commitmentEnterprisePlan         P30D
 1  http://example.com/serviceLPlanHighAvailability  http://example.com/commitmentLPlanHighAvailability  P30D
 2  http://example.com/serviceLPlanSingleNode        http://example.com/commitmentLPlanSingleNode        P30D
 3  http://example.com/serviceMPlanHighAvailability  http://example.com/commitmentMPlanHighAvailability  P30D
 4  http://example.com/serviceMPlanSingleNode        http://example.com/commitmentMPlanSingleNode        P30D
 5  http://example.com/serviceSPlanHighAvailability  http://example.com/commitmentSPlanHighAvailability  P30D
 6  http://example.com/serviceSPlanSingleNode        http://example.com/commitmentSPlanSingleNode        P30D


- Alibaba SLA

In [47]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ7_period_time.rq" 
run_sparql_query(file_path, query_file, file_format)

    service                                 duty                                          duration
--  --------------------------------------  --------------------------------------------  ----------
 0  http://example.com/instanceEcsService   http://example.com/uptimeCommitmentInstance   P30D
 1  http://example.com/multiZoneEcsService  http://example.com/uptimeCommitmentMultiZone  P30D


### CQ8: How are current values of a **service property computed**? 

- Amazon EC2 SLA

In [48]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ8_SLI_expression.rq" 
run_sparql_query(file_path, query_file, file_format)

    leftOperand                                 metricLabel                expression                                     unit                                interval
--  ------------------------------------------  -------------------------  ---------------------------------------------  ----------------------------------  ----------------------------------
 0  http://example.com/monthlyUptimePercentage  Monthly Uptime Percentage  100 - (100 * ?downtimeMinutes / totalMinutes)  http://qudt.org/vocab/unit#Percent  http://example.com/monthlyInterval


- Cedalo SLA

In [49]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/competency_questions/CQ8_SLI_expression.rq" 
run_sparql_query(file_path, query_file, file_format)

    leftOperand                                 metricLabel                expression                                unit                                interval
--  ------------------------------------------  -------------------------  ----------------------------------------  ----------------------------------  ----------------------------------
 0  http://example.com/monthlyUptimePercentage  Monthly Uptime Percentage  100 - (100 * ?inactivityMinutes / 43200)  http://qudt.org/vocab/unit#Percent  http://example.com/monthlyInterval


- Alibaba SLA

In [50]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/competency_questions/CQ8_SLI_expression.rq" 
run_sparql_query(file_path, query_file, file_format)

    leftOperand                                                     metricLabel                                                 expression                                                                unit                                interval
--  --------------------------------------------------------------  ----------------------------------------------------------  ------------------------------------------------------------------------  ----------------------------------  ----------------------------------
 0  http://example.com/monthlyUptimePercentageInstanceUnavailable   Monthly Uptime Percentage - Instance Unavailable            100 * ((?serviceCycleMinutes - ?downtimeMinutes) / ?serviceCycleMinutes)  http://qudt.org/vocab/unit#Percent  http://example.com/monthlyInterval
 1  http://example.com/monthlyUptimePercentageMultiZoneUnavailable  Monthly Uptime Percentage - Multi-zone Service Unavailable  100 * ((?serviceCycleMinutes - ?downtimeMinutes) / ?serviceCycleMinutes)  http:

------------

## Unfair Terms
The Unfair Contract Terms Directive (UCTD) stipulates that a contractual term is considered unfair if it has not been individually negotiated and, contrary to the requirements of good faith, it causes a significant imbalance in the parties’ rights and obligations under the contract, to the detriment of the consumer.

All the queries used for detecting potentially unfair terms are located in the `sparql_queries/unfair_terms` directory. Additionally, we provide an explanatory README.md detailing the design decisions behind these queries.

In this section, we execute the defined **8 unfairness detection queries** to identify potentially unfair clauses. Each query is applied to the SLAs of Amazon and Cedalo, as well as to the ToS and SLA of Alibaba.

**Note:** The output is empty if there are no potentially unfair terms. It only returns results when the agreement contains one or more clauses that may be considered unfair.

#### Potentially Unfair Terms in **Arbitration**

Does the agreement include any arbitration clauses? If so, are these clauses related to the limitation of litigation rights?


- Amazon EC2 SLA

In [52]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/unfair_terms/arbitration.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [53]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/unfair_terms/arbitration.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA and ToS

In [54]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/unfair_terms/arbitration.rq" 
run_sparql_query(file_path, query_file, file_format)

In [115]:
file_path = "examples/alibaba/alibaba_tos.ttl"
query_file = "../sparql_queries/unfair_terms/arbitration.rq" 
run_sparql_query(file_path, query_file, file_format)

#### Potentially Unfair Terms in **Choice of Law**

Are there any dispute resolution clauses that apply a law other than the law of the consumer’s place of residence?


- Amazon EC2 SLA

In [55]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/unfair_terms/choice_of_law.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [56]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/unfair_terms/choice_of_law.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA and ToS

In [57]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/unfair_terms/choice_of_law.rq" 
run_sparql_query(file_path, query_file, file_format)

In [116]:
file_path = "examples/alibaba/alibaba_tos.ttl"
query_file = "../sparql_queries/unfair_terms/choice_of_law.rq" 
run_sparql_query(file_path, query_file, file_format)

#### Potentially Unfair Terms in Content Removal

Is content removal allowed without providing a reason, without prior notice, or without the possibility of recovery?

- Amazon EC2 SLA

In [58]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/unfair_terms/content_removal.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [59]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/unfair_terms/content_removal.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA and ToS

In [60]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/unfair_terms/content_removal.rq" 
run_sparql_query(file_path, query_file, file_format)

In [117]:
file_path = "examples/alibaba/alibaba_tos.ttl"
query_file = "../sparql_queries/unfair_terms/content_removal.rq" 
run_sparql_query(file_path, query_file, file_format)

    permission                             action                        assignee                    target
--  -------------------------------------  ----------------------------  --------------------------  ---------------------------
 0  http://example.com/permProductChanges  https://w3id.org/tosl/remove  http://example.com/Alibaba  http://example.com/services
 1  http://example.com/permDataChanges     https://w3id.org/tosl/remove  http://example.com/Alibaba  http://example.com/content


#### Potentially Unfair Terms in Contract by Use

Is the consumer considered to have accepted the contract merely by using the service or through implied consent?


- Amazon EC2 SLA

In [61]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/unfair_terms/contract_by_use.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [62]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/unfair_terms/contract_by_use.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA and ToS

In [63]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/unfair_terms/contract_by_use.rq" 
run_sparql_query(file_path, query_file, file_format)

In [118]:
file_path = "examples/alibaba/alibaba_tos.ttl"
query_file = "../sparql_queries/unfair_terms/contract_by_use.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                                action                         assignee                     target
--  --------------------------------------------------  -----------------------------  ---------------------------  ----------------------------------------
 0  http://example.com/dutyImplicitConsent              https://w3id.org/tosl/consent  http://example.com/customer  http://example.com/AlibabaTermsOfService
 1  http://example.com/dutyImplicitConsentAmendedTerms  https://w3id.org/tosl/consent  http://example.com/customer  http://example.com/AlibabaTermsOfService


#### Potentially Unfair Terms in Jurisdiction

Does the dispute have to be settled in the courts of a city or country other than the consumer’s place of residence?


- Amazon EC2 SLA

In [64]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/unfair_terms/jurisdiction.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [65]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/unfair_terms/jurisdiction.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA and ToS

In [66]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/unfair_terms/jurisdiction.rq" 
run_sparql_query(file_path, query_file, file_format)

In [119]:
file_path = "examples/alibaba/alibaba_tos.ttl"
query_file = "../sparql_queries/unfair_terms/jurisdiction.rq" 
run_sparql_query(file_path, query_file, file_format)

#### Potentially Unfair Terms in Limitation of Liability

Does the supplier exclude or limit their liability or any of their contractual obligations?


- Amazon EC2 SLA

In [67]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/unfair_terms/limitation_of_liability.rq" 
run_sparql_query(file_path, query_file, file_format)

    liability                                       description                                                                                                                                                      limitationOn                     type                                     liableParty
--  ----------------------------------------------  ---------------------------------------------------------------------------------------------------------------------------------------------------------------  -------------------------------  ---------------------------------------  -------------------------
 0  http://example.com/liabilitySuspension          Suspension or termination of EC2 service due to breach of AWS terms by the Customer.                                                                             http://example.com/amazonEC2SLA  https://w3id.org/tosl/breachOfContract   http://example.com/amazon
 1  http://example.com/liabilityNonMeasuredFactors  Other factors affecting

- Cedalo SLA

In [ ]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/unfair_terms/limitation_of_liability.rq" 
run_sparql_query(file_path, query_file, file_format)

    liability                                   description                                                                                    limitationOn                  type                                       liableParty
--  ------------------------------------------  ---------------------------------------------------------------------------------------------  ----------------------------  -----------------------------------------  -------------------------
 0  http://example.com/liabilityBreachContract  Situations where Cedalo suspends access due to violations of the Customer’s obligations.       http://example.com/cedaloSLA  https://w3id.org/tosl/breachOfContract     http://example.com/cedalo
 1  http://example.com/liabilityForceMajeure    Unavailability due to force majeure, DDoS attacks, or similar events beyond Cedalo’s control.  http://example.com/cedaloSLA  https://w3id.org/tosl/anyIndirectDamage    http://example.com/cedalo
 2  http://example.com/liabilityForceMajeure  

- Alibaba SLA and ToS

In [ ]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/unfair_terms/limitation_of_liability.rq" 
run_sparql_query(file_path, query_file, file_format)

    liability                                       description                                                                                                            limitationOn                      type                                     liableParty
--  ----------------------------------------------  ---------------------------------------------------------------------------------------------------------------------  --------------------------------  ---------------------------------------  --------------------------
 0  http://example.com/liabilitySuspensionContract  Suspension or termination of ECS Service due to breach of Alibaba Cloud terms by the Customer.                         http://example.com/alibabaECSSLA  https://w3id.org/tosl/breachOfContract   http://example.com/alibaba
 1  http://example.com/liabilityIllegalUse          Unavailability caused by illegal or unlawful use of the Service, or breach of Alibaba Cloud terms and conditions.      http://example.com/alibab

In [120]:
file_path = "examples/alibaba/alibaba_tos.ttl"
query_file = "../sparql_queries/unfair_terms/limitation_of_liability.rq" 
run_sparql_query(file_path, query_file, file_format)

#### Potentially Unfair Terms in Unilateral Change

Are changes made unilaterally without justification or prior notice?


- Amazon EC2 SLA

In [68]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/unfair_terms/change.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [69]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/unfair_terms/change.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA and ToS

In [70]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/unfair_terms/change.rq" 
run_sparql_query(file_path, query_file, file_format)

    permission                                    description                                                                                                                                                                                   action                              target
--  --------------------------------------------  --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  ----------------------------------  --------------------------------
 0  http://example.com/slaModificationPermission  Alibaba reserves the right to unilaterally modify the terms of this SLA by posting an amended version on the Alibaba Cloud website. Continued use of the Service shall be deemed acceptance.  http://www.w3.org/ns/odrl/2/modify  http://example.com/alibabaECSSLA


In [121]:
file_path = "examples/alibaba/alibaba_tos.ttl"
query_file = "../sparql_queries/unfair_terms/change.rq" 
run_sparql_query(file_path, query_file, file_format)

    permission                                           description                                                                                                                                                          action                              target
--  ---------------------------------------------------  -------------------------------------------------------------------------------------------------------------------------------------------------------------------  ----------------------------------  ----------------------------------------
 0  http://example.com/permAmendTerms                    Alibaba may amend and restate the Product Terms by posting them on the Platform.                                                                                     http://www.w3.org/ns/odrl/2/modify  http://example.com/AlibabaTermsOfService
 1  http://example.com/permProductChanges                Alibaba may launch, change, upgrade, impose conditions on, suspend, or stop offe

#### Potentially Unfair Terms in Unilateral Termination

Is it permitted to terminate the contract unilaterally, without justification or prior notice?


- Amazon EC2 SLA

In [73]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/unfair_terms/termination.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [72]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/unfair_terms/termination.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA and ToS

In [71]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/unfair_terms/termination.rq" 
run_sparql_query(file_path, query_file, file_format)

In [122]:
file_path = "examples/alibaba/alibaba_tos.ttl"
query_file = "../sparql_queries/unfair_terms/termination.rq" 
run_sparql_query(file_path, query_file, file_format)

    permission                             description                                                                                                                                                          action                           assignee                    target
--  -------------------------------------  -------------------------------------------------------------------------------------------------------------------------------------------------------------------  -------------------------------  --------------------------  ---------------------------
 0  http://example.com/permProductChanges  Alibaba may launch, change, upgrade, impose conditions on, suspend, or stop offering any Product/features, including sign-on and access/URLs, without prior notice.  https://w3id.org/tosl/terminate  http://example.com/Alibaba  http://example.com/services
 1  http://example.com/permDataChanges     Alibaba may relocate, suspend, or cease operations at any data center, at its discretio

------------

## Normative Analysis

In this section, we calculate the total number of **obligations**, **permissions**, and **prohibitions** defined in the agreement. We also distinguish which of these normative elements apply to the **provider** and which apply to the **customer**.


**General information:** Total Number of Rules in the Agreements
- Amazon EC2 SLA

In [75]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_rules.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalElements
--  ---------------
 0               12


- Cedalo SLA

In [76]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_rules.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalElements
--  ---------------
 0               37


- Alibaba SLA

In [77]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_rules.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalElements
--  ---------------
 0               13


## Permissions and Total Number of Permissions

- Amazon EC2 SLA

In [78]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalPermissions
--  ------------------
 0                   0


In [79]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [80]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalPermissions
--  ------------------
 0                   1


In [81]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

    permission                                actions                             targets                                                                                                                                                                                                                                                                                                                       assignee
--  ----------------------------------------  ----------------------------------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  -------------------------
 0  http://example.com/permissionMaintenance  http://www.w3.org/ns/odrl/2/modify  http://example.com/serviceSPlanSingleNode, http://example.com/serviceSPlanHighAvail

- Alibaba SLA

In [82]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalPermissions
--  ------------------
 0                   1


In [83]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

    permission                                    actions                             targets                           assignee
--  --------------------------------------------  ----------------------------------  --------------------------------  --------------------------
 0  http://example.com/slaModificationPermission  http://www.w3.org/ns/odrl/2/modify  http://example.com/alibabaECSSLA  http://example.com/alibaba


#### Provider Permissions

- Amazon EC2 SLA

In [85]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [86]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

    permission                                actions                             targets                                                                                                                                                                                                                                                                                                                       assignee
--  ----------------------------------------  ----------------------------------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  -------------------------
 0  http://example.com/permissionMaintenance  http://www.w3.org/ns/odrl/2/modify  http://example.com/serviceSPlanSingleNode, http://example.com/serviceSPlanHighAvail

- Alibaba SLA

In [87]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

    permission                                    actions                             targets                           assignee
--  --------------------------------------------  ----------------------------------  --------------------------------  --------------------------
 0  http://example.com/slaModificationPermission  http://www.w3.org/ns/odrl/2/modify  http://example.com/alibabaECSSLA  http://example.com/alibaba


#### Customer Permissions

- Amazon EC2 SLA

In [88]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [89]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA

In [90]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_permissions.rq" 
run_sparql_query(file_path, query_file, file_format)

## Duties and Total Number of Duties

- Amazon EC2 SLA

In [91]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalDuties
--  -------------
 0             12


In [92]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                           actions                                 targets                                assignee
--  ---------------------------------------------  --------------------------------------  -------------------------------------  ---------------------------
 0  http://example.com/commitmentEC2Region         https://w3id.org/tosla/guarantee        http://example.com/ec2RegionService    http://example.com/amazon
 1  http://example.com/customerClaimEC2Region      https://w3id.org/tosl/claim             http://example.com/ec2RegionService    http://example.com/customer
 2  http://example.com/compensationEC2Region10     http://www.w3.org/ns/odrl/2/compensate  http://example.com/ec2RegionService    http://example.com/amazon
 3  http://example.com/compensationEC2Region30     http://www.w3.org/ns/odrl/2/compensate  http://example.com/ec2RegionService    http://example.com/amazon
 4  http://example.com/compensationEC2Region100    http://www.w3.org/ns/odrl/

- Cedalo SLA

In [93]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalDuties
--  -------------
 0             36


In [94]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                                      actions                                 targets                                                                                                                                                                                                                                                                                                                       assignee
--  --------------------------------------------------------  --------------------------------------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  -----------------------------------------
 0  http://example.com/commitmentSPlanSingleNode              https://w3id.org/tosla/guarantee        http://

- Alibaba SLA

In [95]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalDuties
--  -------------
 0             12


In [96]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                            actions                                 targets                                 assignee
--  ----------------------------------------------  --------------------------------------  --------------------------------------  ---------------------------
 0  http://example.com/uptimeCommitmentInstance     https://w3id.org/tosla/guarantee        http://example.com/instanceEcsService   http://example.com/alibaba
 1  http://example.com/customerClaimInstance        https://w3id.org/tosl/claim             http://example.com/instanceEcsService   http://example.com/customer
 2  http://example.com/compensationInstance10       http://www.w3.org/ns/odrl/2/compensate  http://example.com/instanceEcsService   http://example.com/alibaba
 3  http://example.com/compensationInstance25       http://www.w3.org/ns/odrl/2/compensate  http://example.com/instanceEcsService   http://example.com/alibaba
 4  http://example.com/compensationInstance100      http://www

#### Provider Duties

- Amazon EC2 SLA

In [97]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                           actions                                 targets                                assignee
--  ---------------------------------------------  --------------------------------------  -------------------------------------  -------------------------
 0  http://example.com/commitmentEC2Region         https://w3id.org/tosla/guarantee        http://example.com/ec2RegionService    http://example.com/amazon
 1  http://example.com/compensationEC2Region10     http://www.w3.org/ns/odrl/2/compensate  http://example.com/ec2RegionService    http://example.com/amazon
 2  http://example.com/compensationEC2Region30     http://www.w3.org/ns/odrl/2/compensate  http://example.com/ec2RegionService    http://example.com/amazon
 3  http://example.com/compensationEC2Region100    http://www.w3.org/ns/odrl/2/compensate  http://example.com/ec2RegionService    http://example.com/amazon
 4  http://example.com/commitmentEC2Instance       https://w3id.org/tosla/guarant

- Cedalo SLA

In [98]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                                      actions                                 targets                                                                                                                                                                                                                                                                                                                       assignee
--  --------------------------------------------------------  --------------------------------------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  -------------------------
 0  http://example.com/commitmentSPlanSingleNode              https://w3id.org/tosla/guarantee        http://example.com/serv

- Alibaba SLA

In [99]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                          actions                                 targets                                 assignee
--  --------------------------------------------  --------------------------------------  --------------------------------------  --------------------------
 0  http://example.com/uptimeCommitmentInstance   https://w3id.org/tosla/guarantee        http://example.com/instanceEcsService   http://example.com/alibaba
 1  http://example.com/compensationInstance10     http://www.w3.org/ns/odrl/2/compensate  http://example.com/instanceEcsService   http://example.com/alibaba
 2  http://example.com/compensationInstance25     http://www.w3.org/ns/odrl/2/compensate  http://example.com/instanceEcsService   http://example.com/alibaba
 3  http://example.com/compensationInstance100    http://www.w3.org/ns/odrl/2/compensate  http://example.com/instanceEcsService   http://example.com/alibaba
 4  http://example.com/uptimeCommitmentMultiZone  https://w3id.org/tosla/gua

#### Customer Duties

- Amazon EC2 SLA

In [100]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                         actions                      targets                              assignee
--  -------------------------------------------  ---------------------------  -----------------------------------  ---------------------------
 0  http://example.com/customerClaimEC2Region    https://w3id.org/tosl/claim  http://example.com/ec2RegionService  http://example.com/customer
 1  http://example.com/customerClaimEC2Instance  https://w3id.org/tosl/claim  http://example.com/ec2RegionService  http://example.com/customer


- Cedalo SLA

In [101]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                                   actions                      targets                                          assignee
--  -----------------------------------------------------  ---------------------------  -----------------------------------------------  -----------------------------------------
 0  http://example.com/customerSPlanSingleNodeClaim        https://w3id.org/tosl/claim  http://example.com/serviceSPlanSingleNode        http://example.com/customerSPlan
 1  http://example.com/customerSPlanHighAvailabilityClaim  https://w3id.org/tosl/claim  http://example.com/serviceSPlanHighAvailability  http://example.com/customerSPlan
 2  http://example.com/customerMPlanSingleNodeClaim        https://w3id.org/tosl/claim  http://example.com/serviceMPlanSingleNode        http://example.com/customerMPlan
 3  http://example.com/customerMPlanHighAvailabilityClaim  https://w3id.org/tosl/claim  http://example.com/serviceMPlanHighAvailability  http://example.com/customerS

- Alibaba SLA

In [102]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_duties.rq" 
run_sparql_query(file_path, query_file, file_format)

    duty                                            actions                          targets                                 assignee
--  ----------------------------------------------  -------------------------------  --------------------------------------  ---------------------------
 0  http://example.com/customerClaimInstance        https://w3id.org/tosl/claim      http://example.com/instanceEcsService   http://example.com/customer
 1  http://example.com/customerClaimMultiZone       https://w3id.org/tosl/claim      http://example.com/multiZoneEcsService  http://example.com/customer
 2  http://example.com/customerCreditUsageInterval  http://www.w3.org/ns/odrl/2/use  http://example.com/serviceCredit        http://example.com/customer


## Prohibitions and Total Number of Prohibitions

- Amazon EC2 SLA

In [103]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalProhibitions
--  -------------------
 0                    0


In [104]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [105]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalProhibitions
--  -------------------
 0                    0


In [106]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA

In [107]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/total_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

      totalProhibitions
--  -------------------
 0                    0


In [108]:
query_file = "../sparql_queries/obligations_permissions_prohibition/get_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

#### Provider Prohibitions

- Amazon EC2 SLA

In [109]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [110]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA

In [111]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_provider_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

#### Customer Prohibitions

- Amazon EC2 SLA

In [112]:
file_path = "examples/amazon/amazon_ec2_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Cedalo SLA

In [113]:
file_path = "examples/cedalo/cedalo_sla_2025.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)

- Alibaba SLA

In [114]:
file_path = "examples/alibaba/alibaba_sla.ttl"
query_file = "../sparql_queries/obligations_permissions_prohibition/get_customer_prohibitions.rq" 
run_sparql_query(file_path, query_file, file_format)